# Remaining Useful Life (RUL) Estimation
**Goal:** estimate how many more charge-discharge cycles a battery has before its State of Health (SOH) falls to the End-of-Life (EOL) threshold.

**Design choices (and why)**
- **EOL = SOH ≤ 0.80** (the common 80 % definition). RUL(cycle) = EOL cycle − current cycle.
- **Only batteries that start healthy (SOH ≥ 0.9) and actually reach EOL** are used (13 batteries). Others (cold-temperature tests whose capacity *rises*, batteries that never degrade in the log) have no meaningful RUL label.
- **Leakage-safe features only:** everything a device can know *at the current cycle* — cycles since first use, measured SOH (from the full-discharge capacity count), recent fade rate, temperature, current. No final-cycle information.
- **Evaluation:** Leave-One-Battery-Out (LOBO): each battery is predicted by models trained on the *other* 12. With so few batteries this is more honest than a single split.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor

CSV = 'cycle_summary_soh_rul_FINAL.csv'
EOL = 0.80   # end-of-life SOH threshold
W   = 10     # window (cycles) for fade-rate estimate
d = pd.read_csv(CSV).sort_values(['battery_id','cycle'])

## 1. Build the RUL dataset

In [ ]:
rows = []
for b, g in d.groupby('battery_id'):
    g = g.reset_index(drop=True)
    if g.soh.iloc[:3].median() < 0.9: continue          # must start healthy
    hit = np.where(g.soh.values <= EOL)[0]
    if not len(hit): continue                           # must reach EOL
    g = g.iloc[:hit[0]+1].copy()                        # keep data up to first EOL crossing
    if len(g) < 20: continue
    g['age']        = g.cycle - g.cycle.iloc[0]                    # cycles since first logged cycle
    g['soh_s']      = g.soh.rolling(5, min_periods=1).mean()       # smoothed measured SOH
    g['slope']      = g.soh_s.diff(W).div(W).fillna(0)             # fade per cycle (last W cycles)
    g['fade_total'] = 1 - g.soh_s
    g['rul']        = g.cycle.iloc[-1] - g.cycle                   # TARGET
    rows.append(g)
D = pd.concat(rows)
print(D.groupby('battery_id').agg(cycles=('cycle','count'), eol_cycle=('cycle','max')).T.to_string())

In [ ]:
FEATS = ['age','soh_s','slope','fade_total','mean_temperature_C','mean_current_A']

## 2. Models compared
| Model | Why |
|---|---|
| **Physics trend** | Straight-line extrapolation of recent SOH to 0.80 — no training, the natural baseline |
| **Linear Regression** | Smallest possible deployable model |
| **XGBoost** | Strong tabular model (needs C export for the ESP32-S3) |
| **MLP** (Multi-Layer Perceptron, 24-12) | Small neural network, deployable via TensorFlow Lite Micro |

In [ ]:
def physics(g):
    out = []
    for i in range(len(g)):
        lo = max(0, i-W)
        if i-lo < 3: out.append(np.nan); continue
        m = np.polyfit(g.age.values[lo:i+1], g.soh_s.values[lo:i+1], 1)[0]
        out.append(max((g.soh_s.values[i]-EOL)/-m, 0) if m < 0 else np.nan)
    return np.array(out)

def make(name):
    if name=='LinReg':  return LinearRegression()
    if name=='XGBoost': return XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.9, random_state=42)
    if name=='MLP':     return make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(24,12), max_iter=2000, early_stopping=True, random_state=42))

names = ['Physics trend','LinReg','XGBoost','MLP']
res = {k: [] for k in names}
for b in D.battery_id.unique():                          # Leave-One-Battery-Out
    tr, te = D[D.battery_id!=b], D[D.battery_id==b]
    p = physics(te); p = np.where(np.isnan(p), np.nanmedian(p) if np.isfinite(np.nanmedian(p)) else 0, p)
    res['Physics trend'].append((b, te.rul.values, p))
    for n in names[1:]:
        m = make(n).fit(tr[FEATS], tr.rul)
        res[n].append((b, te.rul.values, m.predict(te[FEATS])))

## 3. Performance matrix (all held-out batteries pooled; errors in cycles)

In [ ]:
rows = []
for n, v in res.items():
    y = np.concatenate([a[1] for a in v]); p = np.concatenate([a[2] for a in v])
    late = y <= 30                                       # last 30 cycles before EOL — where accuracy matters most
    rows.append({'Model': n,
                 'MAE (cycles)': np.abs(y-p).mean(),
                 'RMSE (cycles)': np.sqrt(((y-p)**2).mean()),
                 'R2': 1 - ((y-p)**2).sum()/((y-y.mean())**2).sum(),
                 'MAE, last 30 cycles': np.abs(y-p)[late].mean()})
perf = pd.DataFrame(rows).round(2); perf

## 4. Predicted vs. true RUL for each held-out battery

In [ ]:
fig, ax = plt.subplots(3, 5, figsize=(20, 9)); ax = ax.ravel()
for i, (b, y, p) in enumerate(res['LinReg']):
    ax[i].plot(y, y, 'k--', lw=1, label='ideal')
    ax[i].plot(y, p, label='LinReg')
    ax[i].plot(y, res['XGBoost'][i][2], alpha=.7, label='XGBoost')
    ax[i].set_title(b); ax[i].set_xlabel('true RUL'); ax[i].set_ylabel('predicted RUL')
ax[0].legend()
for j in range(len(res['LinReg']), len(ax)): ax[j].axis('off')
plt.tight_layout(); plt.show()

## 5. How to read the results
- The physics trend baseline is poor because SOH is noisy cycle to cycle, so a short-window slope is unreliable.
- Learned models are far better but still only moderate: **R² ≈ 0.5–0.6, typical error ≈ 15–18 cycles** (batteries live 40–125 cycles to EOL).
- The honest limit: **13 batteries, different test protocols, and a label that depends on where each test happened to stop.** Treat RUL as a rough trend indicator with uncertainty, not a precise countdown.
- Deployment: Linear Regression is a handful of multiply-adds; the MLP goes through TensorFlow Lite Micro; XGBoost needs a C export. **The simplest model is nearly as good, so it is the practical choice.**
- The RUL input `soh_s` is the *measured* SOH from the full-discharge capacity count (cascaded SOH → RUL).